In [2]:
import os
import gc

import numpy as np
import pandas as pd
import seaborn as sns
import missingno as msno
sns.set(style='darkgrid')
import matplotlib.pyplot as plt
from pathlib import Path
import joblib

print("Working directory:", Path.cwd())

data_list = [pd.read_csv(csv) for csv in Path("../fixed_data/").iterdir()]
data = pd.concat(data_list)

# Deleting dataframes after concating to save memory
for d in data_list: del d
gc.collect()

# Renaming the columns by removing leading/trailing whitespace
col_names = {col: col.strip() for col in data.columns}
data.rename(columns = col_names, inplace = True)
data.columns

print("done entering data")

Working directory: /home/weluv/CMPE-272/src
done entering data


In [3]:
for col in data.select_dtypes(include='float').columns:
    s = data[col]
    
    # 1. Skip columns containing infinite values (inf cannot become an int)
    if np.isinf(s).any():
        continue
        
    # 2. Check if the non-null values are all whole numbers
    non_null = s.dropna()
    if not non_null.empty and non_null.mod(1).eq(0).all():
        # 3. Use 'Int64' (nullable) if there are NaNs, otherwise standard 'int64'
        if s.isna().any():
            data[col] = s.astype('Int64')
        else:
            data[col] = s.astype('int64')

data.drop(columns=['[ZoneTransfer]'], errors='ignore', inplace=True)
data['Label'] = data['Label'].astype('object')

data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2100816 entries, 0 to 371748
Data columns (total 84 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   Flow ID                     object 
 1   Src IP                      object 
 2   Src Port                    Int64  
 3   Dst IP                      object 
 4   Dst Port                    Int64  
 5   Protocol                    Int64  
 6   Timestamp                   object 
 7   Flow Duration               Int64  
 8   Total Fwd Packet            Int64  
 9   Total Bwd packets           Int64  
 10  Total Length of Fwd Packet  Int64  
 11  Total Length of Bwd Packet  Int64  
 12  Fwd Packet Length Max       Int64  
 13  Fwd Packet Length Min       Int64  
 14  Fwd Packet Length Mean      float64
 15  Fwd Packet Length Std       float64
 16  Bwd Packet Length Max       Int64  
 17  Bwd Packet Length Min       Int64  
 18  Bwd Packet Length Mean      float64
 19  Bwd Packet Length Std      

In [4]:
pd.options.display.max_columns = 90
data

,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Total Fwd Packet,Total Bwd packets,Total Length of Fwd Packet,Total Length of Bwd Packet,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Packet Length Min,Packet Length Max,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWR Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Fwd Segment Size Avg,Bwd Segment Size Avg,Fwd Bytes/Bulk Avg,Fwd Packet/Bulk Avg,Fwd Bulk Rate Avg,Bwd Bytes/Bulk Avg,Bwd Packet/Bulk Avg,Bwd Bulk Rate Avg,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,FWD Init Win Bytes,Bwd Init Win Bytes,Fwd Act Data Pkts,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,8.6.0.1-8.0.6.4-0-0-0,8.6.0.1,0,8.0.6.4,0,0,06/07/2017 01:59:05 PM,119945515,150,0,0,0,0,0,0.000000,0.000000,0,0,0.000000,0.000000,0.000000e+00,1.250568,8.050035e+05,5.277837e+06,60127797,0,119945515,8.050035e+05,5.277837e+06,60127797,0,0,0.000000e+00,0.000000e+00,0,0,0,0,0,0,0,0,1.250568,0.000000,0,0,0.000000,0.000000,0.000000,0,0,0,0,0,0,0,0,0.000000,0.000000,0.000000,0.000000,0,0,0,0,0,0,1,0,0,0,0,0,0,0,6.909777e+06,1.165368e+07,20364643,6,2.436882e+07,2.431326e+07,60127797,5702188,BENIGN
1,192.168.10.50-192.168.10.3-33898-389-6,192.168.10.50,33898,192.168.10.3,389,6,06/07/2017 01:59:00 PM,113095465,48,24,9668,10012,403,0,201.416667,203.548294,923,316,417.166667,231.080951,1.740123e+02,0.636630,1.592894e+06,4.597265e+06,16482821,3,113095465,2.406286e+06,5.491986e+06,16482821,3,113095083,4.917178e+06,7.084368e+06,16483210,3,24,24,0,0,1536,768,0.424420,0.212210,0,923,273.333333,234.977733,55214.535211,0,0,0,48,72,0,0,0,0.500000,273.333333,201.416667,417.166667,0,0,0,0,0,0,0,134,0,139,571,2079,23,32,2.039855e+05,5.758373e+05,1629110,379,1.382116e+07,4.277541e+06,16482821,6737603,BENIGN
2,192.168.10.50-192.168.10.3-33904-389-6,192.168.10.50,33904,192.168.10.3,389,6,06/07/2017 01:59:00 PM,113473706,68,40,11364,12718,403,0,167.117647,171.919413,1139,126,317.950000,208.261294,2.122254e+02,0.951762,1.060502e+06,3.813685e+06,16482853,2,113473706,1.693637e+06,4.719143e+06,16482853,2,113427590,2.908400e+06,5.922355e+06,16483212,3,40,40,0,0,2176,1280,0.599258,0.352505,0,1139,222.981481,199.175449,39670.859467,0,0,0,80,108,0,0,0,0.588235,222.981481,167.117647,317.950000,0,0,0,0,0,0,0,105,0,117,390,2081,39,32,1.783269e+05,5.034269e+05,1424245,325,1.384682e+07,4.229413e+06,16482853,6945512,BENIGN
3,192.168.10.19-192.168.10.3-32794-389-6,192.168.10.19,32794,192.168.10.3,389,6,06/07/2017 02:00:43 PM,118034439,106,61,17896,27110,1313,0,168.830189,225.320575,3069,0,444.426230,470.042036,3.812955e+02,1.414841,7.110508e+05,4.579007e+06,49024651,0,118034439,1.124138e+06,5.726832e+06,49024651,1,117995477,1.966591e+06,7.492251e+06,49025137,2,58,58,0,0,3408,1968,0.898043,0.516798,0,3069,269.497006,360.120058,129686.456316,0,4,0,116,165,0,0,0,0.575472,269.497006,168.830189,444.426230,0,0,0,0,0,0,0,107,0,162,29200,2078,58,32,1.675640e+06,1.059676e+06,3343206,636188,2.192323e+07,1.681534e+07,49024651,6604342,BENIGN
4,192.168.10.19-192.168.10.3-44766-3268-6,192.168.10.19,44766,192.168.10.3,3268,6,06/07/2017 02:00:44 PM,116446503,72,47,13380,15294,1313,0,185.833333,247.305080,3069,0,325.404255,517.847328,2.462418e+02,1.021928,9.868348e+05,5.722644e+06,49206703,0,116446503,1.640092e+06,7.324638e+06,49206703,1,116446413,2.531444e+06,9.013164e+06,49244823,3,44,44,0,0,2

In [5]:
# data cleaning

data.drop_duplicates(inplace = True)
data.replace([np.inf, -np.inf], np.nan, inplace = True)

# Filling missing values with median
med_flow_bytes = data['Flow Bytes/s'].median()
med_flow_packets = data['Flow Packets/s'].median()
data['Flow Bytes/s'].fillna(med_flow_bytes, inplace = True)
data['Flow Packets/s'].fillna(med_flow_packets, inplace = True)

print("done")
data.info()

done
<class 'pandas.core.frame.DataFrame'>
Index: 2097672 entries, 0 to 371748
Data columns (total 84 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   Flow ID                     object 
 1   Src IP                      object 
 2   Src Port                    Int64  
 3   Dst IP                      object 
 4   Dst Port                    Int64  
 5   Protocol                    Int64  
 6   Timestamp                   object 
 7   Flow Duration               Int64  
 8   Total Fwd Packet            Int64  
 9   Total Bwd packets           Int64  
 10  Total Length of Fwd Packet  Int64  
 11  Total Length of Bwd Packet  Int64  
 12  Fwd Packet Length Max       Int64  
 13  Fwd Packet Length Min       Int64  
 14  Fwd Packet Length Mean      float64
 15  Fwd Packet Length Std       float64
 16  Bwd Packet Length Max       Int64  
 17  Bwd Packet Length Min       Int64  
 18  Bwd Packet Length Mean      float64
 19  Bwd Packet Length Std 

/tmp/ipykernel_28703/3742571967.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['Flow Bytes/s'].fillna(med_flow_bytes, inplace = True)
/tmp/ipykernel_28703/3742571967.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplac

In [6]:
# Creating a dictionary that maps each label to its attack type
attack_map = {
    'BENIGN': 'BENIGN',
    'DDoS': 'DDoS',
    'DoS Hulk': 'DoS',
    'DoS GoldenEye': 'DoS',
    'DoS slowloris': 'DoS',
    'DoS Slowhttptest': 'DoS',
    'PortScan': 'Port Scan',
    'FTP-Patator': 'Brute Force',
    'SSH-Patator': 'Brute Force',
    'Bot': 'Bot',
    'Web Attack � Brute Force': 'Web Attack',
    'Web Attack � XSS': 'Web Attack',
    'Web Attack � Sql Injection': 'Web Attack',
    'Infiltration': 'Infiltration',
    'Heartbleed': 'Heartbleed'
}

# Creating a new column 'Attack Type' in the DataFrame based on the attack_map dictionary
if 'Label' in data.columns:
    data['Attack Type'] = data['Label'].map(attack_map)

    data.drop('Label', axis = 1, inplace = True)

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
data['Attack Number'] = le.fit_transform(data['Attack Type'])

print("done mapping data")

done mapping data


In [7]:
# data processing

for col in data.columns:
    col_type = data[col].dtype
    if col_type != object:
        c_min = data[col].min()
        c_max = data[col].max()
        # Downcasting float64 to float32
        if str(col_type).find('float') >= 0 and c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
            data[col] = data[col].astype(np.float32)

        # Downcasting int64 to int32
        elif str(col_type).find('int') >= 0 and c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
            data[col] = data[col].astype(np.int32)
            
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2097672 entries, 0 to 371748
Data columns (total 85 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   Flow ID                     object 
 1   Src IP                      object 
 2   Src Port                    Int64  
 3   Dst IP                      object 
 4   Dst Port                    Int64  
 5   Protocol                    Int64  
 6   Timestamp                   object 
 7   Flow Duration               Int64  
 8   Total Fwd Packet            Int64  
 9   Total Bwd packets           Int64  
 10  Total Length of Fwd Packet  Int64  
 11  Total Length of Bwd Packet  Int64  
 12  Fwd Packet Length Max       Int64  
 13  Fwd Packet Length Min       Int64  
 14  Fwd Packet Length Mean      float32
 15  Fwd Packet Length Std       float32
 16  Bwd Packet Length Max       Int64  
 17  Bwd Packet Length Min       Int64  
 18  Bwd Packet Length Mean      float32
 19  Bwd Packet Length Std      

In [8]:
# Dropping columns with only one unique value
num_unique = data.nunique()
one_variable = num_unique[num_unique == 1]
not_one_variable = num_unique[num_unique > 1].index

dropped_cols = one_variable.index
data = data[not_one_variable]

print('Dropped columns:')
dropped_cols

Dropped columns:


Index([], dtype='object')

In [9]:
# Columns after removing non variant columns
data.columns

Index(['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol',
       'Timestamp', 'Flow Duration', 'Total Fwd Packet', 'Total Bwd packets',
       'Total Length of Fwd Packet', 'Total Length of Bwd Packet',
       'Fwd Packet Length Max', 'Fwd Packet Length Min',
       'Fwd Packet Length Mean', 'Fwd Packet Length Std',
       'Bwd Packet Length Max', 'Bwd Packet Length Min',
       'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s',
       'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max',
       'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std',
       'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean',
       'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags',
       'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
       'Packet Length Min', 'Packet Length Max', 'Packet Length Mean',
       'Packet Length Std', 'Packet Len

In [18]:
# drop columns that would leak data
columns_to_drop = ['Flow ID', 'Src IP', 'Dst IP', 'Src Port', 'Dst Port', 'Timestamp']
data = data.drop(columns=columns_to_drop, errors='ignore')

# Keep only TCP flows
print("Before TCP filter:", len(data))
print(data['Protocol'].value_counts())

data = data[data['Protocol'] == 6].copy()

print("After TCP filter:", len(data))
print(data['Attack Type'].value_counts())   # check each family survived

# drop more faulty columns
data = data.drop(columns=['Protocol', 'FWD Init Win Bytes', 'Bwd Init Win Bytes',
                          'Fwd Seg Size Min', 'Packet Length Min',
                          'Fwd Packet Length Min', 'Bwd Packet Length Min'],
                 errors='ignore')
data.columns

Before TCP filter: 2097672
Protocol
6     1096406
17     999637
0        1628
Name: count, dtype: Int64
After TCP filter: 1096406
Attack Type
BENIGN          653290
DoS             171779
Port Scan       159147
DDoS             95123
Brute Force       6953
Bot                738
Infiltration        32
Heartbleed          11
Name: count, dtype: int64


Index(['Flow Duration', 'Total Fwd Packet', 'Total Bwd packets',
       'Total Length of Fwd Packet', 'Total Length of Bwd Packet',
       'Fwd Packet Length Max', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s',
       'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max',
       'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std',
       'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean',
       'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags',
       'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
       'Packet Length Max', 'Packet Length Mean', 'Packet Length Std',
       'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count',
       'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count',
       'CWR Flag Count', 'ECE Flag 

In [19]:
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Config
CV_FOLDS = 5
SEED = 42

# Data prep. 
# replace y with the commented out stuff if we want multi-class
"""
y = le.fit_transform(data["Attack Type"].astype(str))
""" 
y = (data["Attack Type"].astype(str).str.upper() != "BENIGN").astype(np.int8).to_numpy()
X = data.drop(columns=["Attack Type", "Attack Number"]).to_numpy(dtype=np.float32)
feature_names = [c for c in data.columns if c not in ("Attack Type", "Attack Number")]
attack_type = data["Attack Type"].to_numpy()   # kept for per-family breakdown
# del data
gc.collect()
 
X_train, X_test, y_train, y_test, _, type_test = train_test_split(
    X, y, attack_type, test_size=0.2, stratify=y, random_state=SEED
)
print(type(X_train))
del X, y
gc.collect()
print(f"Train {X_train.shape}, test {X_test.shape}, attack rate {y_train.mean():.3f}")

# Model: no scaler / PCA / SMOTE needed
model = HistGradientBoostingClassifier(
    max_iter=75,
    learning_rate=0.1,
    class_weight="balanced",   # handles imbalance without synthetic samples
    early_stopping=True,       # internal validation split stops training when it plateaus
    validation_fraction=0.1,
    n_iter_no_change=10,
    random_state=SEED,
    verbose=1
)
 
# Stratified k-fold cross-validation
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
res = cross_validate(
    model, X_train, y_train, cv=cv,
    scoring=["accuracy", "f1", "precision", "recall"],
    n_jobs=1,   # each fit already uses all cores; parallel folds would multiply memory
)
for m in ["accuracy", "f1", "precision", "recall"]:
    s = res[f"test_{m}"]
    print(f"CV {m:<9}: {np.round(s, 4)}  mean={s.mean():.4f} (+/- {s.std():.4f})")
 
# Final model on the full training set, scored on the untouched test set
model.fit(X_train, y_train)
print(f"\nStopped at {model.n_iter_} iterations")

# dump columns for use later
joblib.dump(model, "hgb_model.joblib")
joblib.dump(le, "label_encoder.joblib")
joblib.dump(feature_names, "feature_columns.joblib")
 
y_pred = model.predict(X_test)
print(f"Test accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Test F1:       {f1_score(y_test, y_pred):.4f}")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["Benign", "Attack"], digits=4))

# Recall per attack family (shows which attacks are hardest to catch)
print("\nDetection rate by original attack type (test set):")
for t in np.unique(type_test):
    mask = type_test == t
    if str(t).upper() == "BENIGN":
        print(f"  {t:<30} n={mask.sum():>8}  correctly benign: {(y_pred[mask] == 0).mean():.4f}")
    else:
        print(f"  {t:<30} n={mask.sum():>8}  detected:         {(y_pred[mask] == 1).mean():.4f}")

<class 'numpy.ndarray'>
Train (877124, 70), test (219282, 70), attack rate 0.404
Binning 0.354 GB of training data: 7.975 s
Binning 0.039 GB of validation data: 0.142 s
Fitting gradient boosted rounds:
Fit 75 trees in 18.590 s, (2325 total leaves)
Time spent computing histograms: 14.999s
Time spent finding best splits:  0.175s
Time spent applying splits:      0.683s
Time spent predicting:           0.168s
Binning 0.354 GB of training data: 2.149 s
Binning 0.039 GB of validation data: 0.140 s
Fitting gradient boosted rounds:
Fit 75 trees in 18.186 s, (2325 total leaves)
Time spent computing histograms: 14.929s
Time spent finding best splits:  0.178s
Time spent applying splits:      0.686s
Time spent predicting:           0.189s
Binning 0.354 GB of training data: 2.142 s
Binning 0.039 GB of validation data: 0.134 s
Fitting gradient boosted rounds:
Fit 75 trees in 18.540 s, (2325 total leaves)
Time spent computing histograms: 14.887s
Time spent finding best splits:  0.179s
Time spent appl

TypeError: '<' not supported between instances of 'str' and 'float'

In [20]:
from sklearn.metrics import roc_auc_score
X_train_df = pd.DataFrame(X_train, columns=feature_names)
aucs = {c: roc_auc_score(y_train, X_train_df[c].fillna(0)) for c in X_train_df.columns}
s = pd.Series(aucs).sub(0.5).abs().sort_values(ascending=False)
print(s.head(15))

Fwd IAT Min                   0.331362
Bwd IAT Total                 0.311841
RST Flag Count                0.301496
Bwd IAT Max                   0.293125
Bwd IAT Mean                  0.290126
Fwd Act Data Pkts             0.276367
PSH Flag Count                0.269228
Bwd Packets/s                 0.267167
Total Length of Fwd Packet    0.263393
Fwd IAT Total                 0.263344
Fwd PSH Flags                 0.262333
ACK Flag Count                0.261057
Bwd PSH Flags                 0.260068
Total Fwd Packet              0.256467
Total Bwd packets             0.254743
dtype: float64


In [16]:
cols = ['RST Flag Count','Fwd Seg Size Min','FWD Init Win Bytes','Packet Length Min',
        'Protocol','SYN Flag Count','Fwd Header Length']

train_df = pd.DataFrame(X_train, columns=feature_names)
train_df['y'] = y_train

print("TRAIN median by class")
print(train_df.groupby('y')[cols].median().T)

TRAIN median by class
y                      0        1
RST Flag Count       0.0      1.0
Fwd Seg Size Min     8.0     24.0
FWD Init Win Bytes   0.0  29200.0
Packet Length Min   33.0      0.0
Protocol            17.0      6.0
SYN Flag Count       0.0      2.0
Fwd Header Length   16.0    172.0
